<a href="https://colab.research.google.com/github/Adamphoenix003/GNN-LinkPrediction/blob/main/GATandSAGE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
print(torch.__version__)
!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -q

In [1]:
import torch
import numpy as np
import pandas as pd
from torch_geometric.data import Data
from torch_geometric.utils import to_undirected
from torch_geometric.transforms import RandomLinkSplit
from sklearn.preprocessing import LabelEncoder

# Paths
content_file = "/content/sample_data/cora.content"
cites_file = "/content/sample_data/cora.cites"

# ---- Load content ----
content = pd.read_csv(content_file, sep='\t', header=None)

paper_ids = content.iloc[:, 0].values
features = torch.tensor(content.iloc[:, 1:-1].values, dtype=torch.float)

labels = LabelEncoder().fit_transform(content.iloc[:, -1])
labels = torch.tensor(labels, dtype=torch.long)

id_map = {j: i for i, j in enumerate(paper_ids)}

# ---- Load cites ----
cites = pd.read_csv(cites_file, sep='\t', header=None)

edges = []
for i in range(len(cites)):
    cited = cites.iloc[i, 0]
    citing = cites.iloc[i, 1]
    if cited in id_map and citing in id_map:
        edges.append([id_map[citing], id_map[cited]])

edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

# Make graph undirected
edge_index = to_undirected(edge_index)

data = Data(x=features, edge_index=edge_index, y=labels)

# Normalize features
data.x = data.x / data.x.sum(1, keepdim=True).clamp(min=1)

print(data)

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708])


In [2]:
transform = RandomLinkSplit(
    num_val=0.05,
    num_test=0.10,
    is_undirected=True,
    neg_sampling_ratio=1.0,
)

train_data, val_data, test_data = transform(data)

print(train_data)

Data(x=[2708, 1433], edge_index=[2, 8976], y=[2708], edge_label=[8976], edge_label_index=[2, 8976])


In [3]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class GraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.dropout = 0.5

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

In [4]:
def decode(z, edge_label_index):
    return (z[edge_label_index[0]] * z[edge_label_index[1]]).sum(dim=1)

In [10]:
def train(model, data, train_data, epochs=300):
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.01,
        weight_decay=5e-4
    )

    criterion = torch.nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        # FULL graph for message passing
        z = model(data.x, data.edge_index)

        out = decode(z, train_data.edge_label_index)
        loss = criterion(out, train_data.edge_label)

        loss.backward()
        optimizer.step()

        if epoch % 50 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

    return model

In [11]:
from sklearn.metrics import roc_auc_score, average_precision_score

@torch.no_grad()
def evaluate(model, data_split):
    model.eval()

    z = model(data.x, data.edge_index)  # FULL graph
    out = decode(z, data_split.edge_label_index)
    preds = torch.sigmoid(out).cpu().numpy()
    labels = data_split.edge_label.cpu().numpy()

    auc = roc_auc_score(labels, preds)
    ap = average_precision_score(labels, preds)

    return auc, ap

In [16]:
model = GraphSAGE(data.num_features, 128)

model = train(model,data, train_data, epochs=300)

val_auc, val_ap = evaluate(model, val_data)
test_auc, test_ap = evaluate(model, test_data)

print("\nValidation AUC:", val_auc)
print("Validation AP :", val_ap)
print("Test AUC:", test_auc)
print("Test AP :", test_ap)

Epoch 0, Loss: 0.7094
Epoch 50, Loss: 0.5270
Epoch 100, Loss: 0.4777
Epoch 150, Loss: 0.4665
Epoch 200, Loss: 0.4686
Epoch 250, Loss: 0.4612

Validation AUC: 0.7304717431219189
Validation AP : 0.6956642598182048
Test AUC: 0.725844258251749
Test AP : 0.7028447574088991


In [55]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv

class GATEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()

        # Multi-head attention
        self.conv1 = GATConv(
            in_channels,
            hidden_channels,
            heads=8,
            dropout=0.6
        )

        # Output layer (no concat)
        self.conv2 = GATConv(
            hidden_channels * 8,
            hidden_channels,
            heads=1,
            concat=False,
            dropout=0.6
        )

    def forward(self, x, edge_index):
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv2(x, edge_index)
        return x

In [56]:
model = GATEncoder(data.num_features, 256)

model = train_link_predictor(model, data, epochs=300)

val_auc, val_ap = evaluate_link_prediction(model, data, split="val")
test_auc, test_ap = evaluate_link_prediction(model, data, split="test")

print("\nValidation AUC:", val_auc)
print("Validation AP :", val_ap)
print("Test AUC:", test_auc)
print("Test AP :", test_ap)

Epoch 0, Loss: 1.3803
Epoch 20, Loss: 1.4335
Epoch 40, Loss: 1.2612
Epoch 60, Loss: 1.1624
Epoch 80, Loss: 1.1261
Epoch 100, Loss: 1.1164
Epoch 120, Loss: 1.0973
Epoch 140, Loss: 1.0688
Epoch 160, Loss: 1.0624
Epoch 180, Loss: 1.0696
Epoch 200, Loss: 1.0756
Epoch 220, Loss: 1.0489
Epoch 240, Loss: 1.0683
Epoch 260, Loss: 1.0508
Epoch 280, Loss: 1.0614

Validation AUC: 0.9336407928407234
Validation AP : 0.9386814382168913
Test AUC: 0.9365280543263397
Test AP : 0.9401347891936622


In [14]:
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures

dataset = Planetoid(
    root='/tmp/Cora',
    name='Cora',
    transform=NormalizeFeatures()
)

data = dataset[0]

Processing...
Done!


In [15]:
from torch_geometric.transforms import RandomLinkSplit

transform = RandomLinkSplit(
    num_val=0.05,
    num_test=0.10,
    is_undirected=True,
    neg_sampling_ratio=1.0
)

train_data, val_data, test_data = transform(data)